# Efficient AI Inference System — LLM serving under concurrent load (Colab T4)

**Goal.** Measure how a vLLM server behaves as concurrent users grow (1 → 10 → 25 → 50 → 100) and
quantify what **continuous batching**, **KV caching** and **quantization** buy you, in TTFT,
inter-token latency (ITL), P95/P99 end-to-end latency, throughput, KV-cache usage and VRAM.

**What came from the two reference repos**

| Idea | Source | How it's used here |
|---|---|---|
| Realistic coding prompts; exact output length via `min_tokens = max_tokens` | systalyze `workload/aws-p50p90-v1-mintok` | Prompts are sliced from that file into windows of **exactly 512 tokens**, sent as token IDs; outputs are held to exactly 128 tokens |
| Prefix caching OFF for the main sweep, temperature 0, one launch line per config saved with results | systalyze `configs/*/launch.sh`, `row.json` | `configs/experiments.yaml` + `command.txt` per config |
| Sweep driver over a concurrency × num-prompts grid, results as one file per case, then aggregate | vllm-benchmarking-studio `combos.yaml`, `run_sweep.py`, `aggregate_result.py` | `infbench/runner.py` → `summary.csv` (resumable) |
| Semaphore-bounded concurrency (`--max-concurrency`) + streaming TTFT/ITL measurement | studio `benchmark_serving.py` / `backend_request_func.py` | `infbench/loadgen.py` (asyncio + aiohttp) |

**Run order:** 1 → 6 once per session, then the experiment cells. Every experiment is resumable: if Colab disconnects, re-run setup and the same cell — finished (config, concurrency) pairs are skipped.

## 1 · Check the GPU
You need a **T4 GPU** runtime (Runtime → Change runtime type).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv

## 2 · Mount Google Drive (results survive disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/efficient-inference'   # code + results live here
RESULTS_DIR = f'{PROJECT_DIR}/results'
import os; os.makedirs(f'{PROJECT_DIR}/infbench', exist_ok=True); os.makedirs(f'{PROJECT_DIR}/configs', exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(PROJECT_DIR)

## 3 · Install
vLLM brings its own PyTorch build. If the cell ends with a message asking to restart, do
**Runtime → Restart session**, then continue from step 2 (skip this cell).

If the latest vLLM fails to start on the T4 (sm 7.5 is an older architecture and support can
change between releases), set `VLLM_VERSION` to a release known to work and re-run.

In [ ]:
VLLM_VERSION = ''   # e.g. '==0.11.0' to pin; '' = latest
!pip install -q "vllm{VLLM_VERSION}" nest_asyncio nvidia-ml-py aiohttp pyyaml
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## 4 · Write the framework to Drive
Each cell below writes one module. You can edit these files directly in Drive (or push them to
your GitHub and replace this section with a `git clone`).

In [ ]:
%%writefile {PROJECT_DIR}/infbench/__init__.py
"""infbench: async load-testing + profiling framework for vLLM on a single GPU (Colab T4)."""
__version__ = "0.1.0"

In [ ]:
%%writefile {PROJECT_DIR}/infbench/workload.py
"""Workload builder.

Idea borrowed from systalyze/vllm-benchmarks (workload/aws-p50p90-v1-mintok):
  * prompts are realistic coding text, not random tokens
  * every request is held to an exact output length with min_tokens == max_tokens

Adapted for a T4 / controlled experiment:
  * input length is FIXED (same token count for every request) by slicing the
    corpus into windows of exactly `isl` token ids and sending the ids directly
    to /v1/completions, so tokenisation can't change the length.
  * every request gets a different window, so prefix caching can't sneak in
    unless an experiment asks for it (shared_prefix mode).
"""
from __future__ import annotations

import json
import random
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path

SYSTALYZE_URL = (
    "https://raw.githubusercontent.com/systalyze/vllm-benchmarks/main/"
    "workload/aws-p50p90-v1-mintok/requests-qwen3-32b.jsonl"
)


@dataclass
class Request:
    prompt_ids: list[int]
    output_len: int


@dataclass
class Corpus:
    texts: list[str]
    output_lengths: list[int] = field(default_factory=list)  # from systalyze, may be empty


def load_corpus(cache_dir: str | Path, url: str = SYSTALYZE_URL) -> Corpus:
    """Download the systalyze workload once (cached), fall back to synthetic text offline."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / Path(url).name
    if not path.exists():
        try:
            print(f"[workload] downloading {url}")
            urllib.request.urlretrieve(url, path)
        except Exception as exc:  # noqa: BLE001
            print(f"[workload] download failed ({exc}); using synthetic corpus")
            return _synthetic_corpus()
    texts, olens = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            texts.append(row["text"])
            olens.append(int(row.get("output_length", 0)))
    print(f"[workload] corpus: {len(texts)} documents from {path.name}")
    return Corpus(texts, olens)


def _synthetic_corpus(n_docs: int = 200, seed: int = 0) -> Corpus:
    rnd = random.Random(seed)
    words = ("def class return import async await tensor cache kernel batch latency token "
             "memory scheduler request queue throughput block page warp thread gpu").split()
    texts = [" ".join(rnd.choice(words) for _ in range(3000)) for _ in range(n_docs)]
    return Corpus(texts, [])


class TokenStream:
    """One long token-id stream built from the corpus; hands out non-overlapping windows."""

    def __init__(self, tokenizer, corpus: Corpus, seed: int = 0):
        docs = list(corpus.texts)
        random.Random(seed).shuffle(docs)
        ids: list[int] = []
        for d in docs:
            ids.extend(tokenizer.encode(d, add_special_tokens=False))
        self.ids = ids
        self.cursor = 0
        print(f"[workload] token stream: {len(ids):,} tokens")

    def view(self, cursor: int) -> "TokenStream":
        """A cheap copy that shares the token ids but starts at `cursor`."""
        v = TokenStream.__new__(TokenStream)
        v.ids, v.cursor = self.ids, cursor % max(len(self.ids), 1)
        return v

    def take(self, n: int) -> list[int]:
        if self.cursor + n > len(self.ids):  # wrap around: reuse is rare and logged
            print("[workload] token stream wrapped; prompts may start repeating")
            self.cursor = 0
        out = self.ids[self.cursor:self.cursor + n]
        self.cursor += n
        return out


def build_fixed(stream: TokenStream, n: int, isl: int, osl: int) -> list[Request]:
    """n requests, each exactly `isl` input tokens and `osl` output tokens, all unique."""
    return [Request(stream.take(isl), osl) for _ in range(n)]


def build_shared_prefix(stream: TokenStream, n: int, prefix_len: int, suffix_len: int,
                        osl: int, prefix: list[int] | None = None) -> list[Request]:
    """n requests sharing one `prefix_len` prefix (think: a long system prompt / RAG
    context) + a unique `suffix_len` tail. Total input length is still fixed."""
    prefix = prefix if prefix is not None else stream.take(prefix_len)
    return [Request(prefix + stream.take(suffix_len), osl) for _ in range(n)]


def build_variable_osl(stream: TokenStream, corpus: Corpus, n: int, isl: int,
                       scale: float = 0.5, cap: int = 256, floor: int = 16,
                       seed: int = 0) -> list[Request]:
    """Fixed input length, but output lengths follow the systalyze distribution
    (p50 ~200 / p90 ~400, scaled down to fit a T4). This is what makes
    continuous batching beat static batching: requests finish at different times."""
    olens = corpus.output_lengths or [128] * 160
    rnd = random.Random(seed)
    reqs = []
    for _ in range(n):
        o = max(floor, min(cap, round(rnd.choice(olens) * scale)))
        reqs.append(Request(stream.take(isl), o))
    return reqs

In [ ]:
%%writefile {PROJECT_DIR}/infbench/server.py
"""Start / stop a vLLM OpenAI-compatible server as a subprocess (Colab has no Docker).

One server per configuration, mirroring systalyze's "one launch.sh per config":
the config dict is translated into `vllm serve` flags, the full command is saved
next to the results so every number is reproducible.
"""
from __future__ import annotations

import os
import re
import shutil
import signal
import subprocess
import sys
import time
import urllib.request
from pathlib import Path


def config_to_args(cfg: dict) -> list[str]:
    """Translate one experiment config into server CLI flags (vLLM or the naive HF server)."""
    if cfg.get("engine", "vllm") == "hf":
        return ["--max-model-len", str(cfg.get("max_model_len", 2048)),
                "--dtype", "half" if cfg.get("dtype", "half") in ("half", "float16") else "float32"]
    a: list[str] = []
    a += ["--dtype", str(cfg.get("dtype", "half"))]  # T4 (sm75) has no bf16
    a += ["--max-model-len", str(cfg.get("max_model_len", 2048))]
    a += ["--gpu-memory-utilization", str(cfg.get("gpu_memory_utilization", 0.85))]
    a += ["--max-num-seqs", str(cfg.get("max_num_seqs", 256))]
    if cfg.get("quantization"):
        a += ["--quantization", str(cfg["quantization"])]
    if cfg.get("kv_cache_dtype"):
        a += ["--kv-cache-dtype", str(cfg["kv_cache_dtype"])]
    # Prefix caching is ON by default in vLLM V1. We turn it OFF for the main
    # sweep (same as systalyze) so it can't hide inside "baseline" numbers.
    a += ["--enable-prefix-caching"] if cfg.get("prefix_caching") else ["--no-enable-prefix-caching"]
    if cfg.get("max_num_batched_tokens"):
        a += ["--max-num-batched-tokens", str(cfg["max_num_batched_tokens"])]
    if cfg.get("enforce_eager"):
        a += ["--enforce-eager"]
    a += [str(x) for x in cfg.get("extra_args", [])]
    return a


class VLLMServer:
    def __init__(self, model: str, args: list[str], log_path: str | Path,
                 port: int = 8000, served_name: str = "bench", engine: str = "vllm"):
        self.model, self.args, self.port, self.served_name = model, args, port, served_name
        self.engine = engine
        self.log_path = Path(log_path)
        self.base_url = f"http://127.0.0.1:{port}"
        self.proc: subprocess.Popen | None = None

    @property
    def command(self) -> list[str]:
        if self.engine == "hf":
            script = str(Path(__file__).with_name("hf_server.py"))
            return [sys.executable, script, "--model", self.model, "--host", "127.0.0.1",
                    "--port", str(self.port), "--served-model-name", self.served_name] + self.args
        exe = shutil.which("vllm")
        head = [exe, "serve", self.model] if exe else \
            [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", self.model]
        return head + ["--host", "127.0.0.1", "--port", str(self.port),
                       "--served-model-name", self.served_name] + self.args

    def start(self, timeout_s: int = 900) -> None:
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        print("[server] " + " ".join(self.command))
        self._log = open(self.log_path, "w")
        self.proc = subprocess.Popen(self.command, stdout=self._log, stderr=subprocess.STDOUT,
                                     start_new_session=True, env=os.environ.copy())
        t0 = time.time()
        while time.time() - t0 < timeout_s:
            if self.proc.poll() is not None:
                raise RuntimeError(f"vLLM exited with code {self.proc.returncode}.\n"
                                   f"--- last log lines ---\n{self.tail(40)}")
            if self._healthy():
                print(f"[server] ready in {time.time() - t0:.0f}s")
                return
            time.sleep(3)
        self.stop()
        raise TimeoutError(f"vLLM not healthy after {timeout_s}s\n{self.tail(40)}")

    def _healthy(self) -> bool:
        try:
            with urllib.request.urlopen(self.base_url + "/health", timeout=2) as r:
                return r.status == 200
        except Exception:  # noqa: BLE001
            return False

    def stop(self, gpu_release_timeout_s: int = 90) -> None:
        if self.proc and self.proc.poll() is None:
            try:
                os.killpg(self.proc.pid, signal.SIGTERM)
                self.proc.wait(timeout=30)
            except Exception:  # noqa: BLE001
                try:
                    os.killpg(self.proc.pid, signal.SIGKILL)
                except Exception:  # noqa: BLE001
                    pass
        if getattr(self, "_log", None):
            self._log.close()
        _wait_gpu_free(gpu_release_timeout_s)
        print("[server] stopped")

    def tail(self, n: int = 30) -> str:
        try:
            return "".join(self.log_path.read_text(errors="ignore").splitlines(True)[-n:])
        except FileNotFoundError:
            return ""

    def kv_capacity(self) -> dict:
        """vLLM prints its KV-cache budget at startup; that number is the hard ceiling
        on how many tokens can be in flight before requests queue or get preempted."""
        text = self.log_path.read_text(errors="ignore") if self.log_path.exists() else ""
        out = {}
        m = re.search(r"GPU KV cache size:\s*([\d,]+)\s*tokens", text)
        if m:
            out["kv_capacity_tokens"] = int(m.group(1).replace(",", ""))
        m = re.search(r"Maximum concurrency for\s*([\d,]+)\s*tokens per request:\s*([\d.]+)x", text)
        if m:
            out["max_concurrency_at_max_len"] = float(m.group(2))
        m = re.search(r"# GPU blocks:\s*(\d+)", text)  # older engines
        if m:
            out["gpu_blocks"] = int(m.group(1))
        # Where the VRAM went (best effort: log wording differs between versions)
        for key, pat in (("mem_weights_gib", r"Model loading took\s*([\d.]+)\s*GiB"),
                         ("mem_kv_cache_gib", r"Available KV cache memory:\s*([\d.]+)\s*GiB"),
                         ("mem_cuda_graphs_gib", r"Graph capturing finished.*?([\d.]+)\s*GiB"),
                         ("mem_weights_gib", r"weights:\s*([\d.]+)\s*MiB")):   # hf_server
            m = re.search(pat, text)
            if m and key not in out:
                v = float(m.group(1))
                out[key] = v / 1024 if "MiB" in pat else v
        m = re.search(r"max_num_batched_tokens=(\d+)", text)
        if m:
            out["engine_max_num_batched_tokens"] = int(m.group(1))
        m = re.search(r"Using (\w+) attention backend", text)
        if m:
            out["attention_backend"] = m.group(1)
        return out

    def __enter__(self):
        self.start()
        return self

    def __exit__(self, *exc):
        self.stop()


def _wait_gpu_free(timeout_s: int, threshold_mib: int = 1500) -> None:
    """Wait until the previous server's CUDA context is gone so the next config
    starts from a clean GPU (otherwise its KV budget is silently smaller)."""
    try:
        import pynvml
        pynvml.nvmlInit()
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
        t0 = time.time()
        while time.time() - t0 < timeout_s:
            used = pynvml.nvmlDeviceGetMemoryInfo(h).used / 2**20
            if used < threshold_mib:
                return
            time.sleep(2)
        print(f"[server] warning: GPU still has {used:.0f} MiB in use")
    except Exception:  # noqa: BLE001
        time.sleep(5)

In [ ]:
%%writefile {PROJECT_DIR}/infbench/monitor.py
"""Background resource sampler: GPU (NVML), CPU/RAM (psutil) and vLLM's own
Prometheus /metrics (KV-cache usage, running/waiting queue, preemptions).

Why scrape /metrics instead of only nvidia-smi?  vLLM pre-allocates a fixed
slice of VRAM for the KV cache at startup (gpu_memory_utilization), so VRAM
"used" is nearly flat during a run. The interesting, *dynamic* memory signal
is what fraction of those KV blocks are actually occupied: vllm:kv_cache_usage_perc.
"""
from __future__ import annotations

import asyncio
import time

import aiohttp

GAUGES = {
    "kv_usage": ("vllm:kv_cache_usage_perc", "vllm:gpu_cache_usage_perc"),
    "running": ("vllm:num_requests_running", "infbench:num_requests_running"),
    "waiting": ("vllm:num_requests_waiting", "infbench:num_requests_waiting"),
}
COUNTERS = {
    "preemptions": ("vllm:num_preemptions_total", "vllm:num_preemptions"),
    "prefix_hits": ("vllm:prefix_cache_hits_total", "vllm:gpu_prefix_cache_hits_total"),
    "prefix_queries": ("vllm:prefix_cache_queries_total", "vllm:gpu_prefix_cache_queries_total"),
    "prompt_tokens": ("vllm:prompt_tokens_total",),
    "generation_tokens": ("vllm:generation_tokens_total",),
}


def parse_prometheus(text: str) -> dict[str, float]:
    """Minimal exposition-format parser: sums a metric over all its label sets."""
    out: dict[str, float] = {}
    for line in text.splitlines():
        if not line or line.startswith("#"):
            continue
        name = line.split("{", 1)[0].split(" ", 1)[0]
        try:
            val = float(line.rsplit(" ", 1)[-1])
        except ValueError:
            continue
        out[name] = out.get(name, 0.0) + val
    return out


def pick(metrics: dict[str, float], names: tuple[str, ...]) -> float | None:
    for n in names:
        if n in metrics:
            return metrics[n]
    return None


async def scrape(session: aiohttp.ClientSession, base_url: str) -> dict[str, float]:
    try:
        async with session.get(base_url + "/metrics", timeout=aiohttp.ClientTimeout(total=3)) as r:
            return parse_prometheus(await r.text())
    except Exception:  # noqa: BLE001
        return {}


def counter_snapshot(m: dict[str, float]) -> dict[str, float]:
    return {k: (pick(m, names) or 0.0) for k, names in COUNTERS.items()}


class ResourceMonitor:
    def __init__(self, base_url: str, interval_s: float = 1.0):
        self.base_url, self.interval = base_url, interval_s
        self.samples: list[dict] = []
        self._task: asyncio.Task | None = None
        self._stop = asyncio.Event()
        self._nvml = None
        try:
            import pynvml
            pynvml.nvmlInit()
            self._nvml = (pynvml, pynvml.nvmlDeviceGetHandleByIndex(0))
        except Exception:  # noqa: BLE001
            print("[monitor] NVML unavailable - GPU columns will be empty")
        try:
            import psutil
            self._psutil = psutil
            psutil.cpu_percent(None)  # prime the counter
        except Exception:  # noqa: BLE001
            self._psutil = None

    def _hw(self) -> dict:
        s: dict = {}
        if self._nvml:
            nv, h = self._nvml
            mem = nv.nvmlDeviceGetMemoryInfo(h)
            s["vram_used_mib"] = mem.used / 2**20
            s["vram_total_mib"] = mem.total / 2**20
            s["gpu_util_pct"] = nv.nvmlDeviceGetUtilizationRates(h).gpu
        if self._psutil:
            s["cpu_pct"] = self._psutil.cpu_percent(None)
            s["ram_used_mib"] = self._psutil.virtual_memory().used / 2**20
        return s

    async def _loop(self, session):
        t0 = time.perf_counter()
        while not self._stop.is_set():
            m = await scrape(session, self.base_url)
            s = {"t": time.perf_counter() - t0, **self._hw()}
            for k, names in GAUGES.items():
                s[k] = pick(m, names)
            self.samples.append(s)
            try:
                await asyncio.wait_for(self._stop.wait(), timeout=self.interval)
            except asyncio.TimeoutError:
                pass

    async def __aenter__(self):
        self._session = aiohttp.ClientSession()
        self.counters_before = counter_snapshot(await scrape(self._session, self.base_url))
        self._task = asyncio.create_task(self._loop(self._session))
        return self

    async def __aexit__(self, *exc):
        self._stop.set()
        await self._task
        after = counter_snapshot(await scrape(self._session, self.base_url))
        self.counter_delta = {k: after[k] - self.counters_before[k] for k in after}
        await self._session.close()

In [ ]:
%%writefile {PROJECT_DIR}/infbench/loadgen.py
"""Asynchronous load generator (streaming, OpenAI /v1/completions).

Two client policies:

* "closed"  - C virtual users. Each user sends its next request the moment the
              previous one finishes (an asyncio.Semaphore of size C), exactly
              like vLLM's benchmark_serving --max-concurrency used by
              vllm-benchmarking-studio. The server is free to continuously
              batch whatever is in flight.
* "waves"   - emulates STATIC batching from the client: send C requests, wait
              for ALL of them, then send the next C. Slots freed by short
              requests sit idle until the longest one in the wave finishes.

Timing is measured per streamed chunk with time.perf_counter():
  TTFT = first chunk - send time
  ITL  = gaps between consecutive chunks
  E2E  = last byte - send time
"""
from __future__ import annotations

import asyncio
import json
import time
from dataclasses import asdict, dataclass, field

import aiohttp

from .workload import Request


@dataclass
class RequestResult:
    idx: int
    ok: bool = False
    error: str = ""
    start: float = 0.0          # seconds since level start
    ttft: float = 0.0           # seconds
    e2e: float = 0.0            # seconds
    itl: list[float] = field(default_factory=list)  # seconds
    prompt_tokens: int = 0
    output_tokens: int = 0
    expected_output_tokens: int = 0

    def to_row(self) -> dict:
        d = asdict(self)
        d["itl"] = [round(x, 6) for x in self.itl]
        return d


async def _send(session: aiohttp.ClientSession, url: str, model: str, req: Request,
                idx: int, t_level0: float) -> RequestResult:
    res = RequestResult(idx=idx, expected_output_tokens=req.output_len)
    payload = {
        "model": model,
        "prompt": req.prompt_ids,          # token ids -> input length is exact
        "max_tokens": req.output_len,
        "min_tokens": req.output_len,      # systalyze "mintok": exact output length
        "temperature": 0.0,
        "stream": True,
        "stream_options": {"include_usage": True},
    }
    t0 = time.perf_counter()
    res.start = t0 - t_level0
    last = None
    n_chunks = 0
    try:
        async with session.post(url, json=payload) as resp:
            if resp.status != 200:
                res.error = f"HTTP {resp.status}: {(await resp.text())[:300]}"
                return res
            async for raw in resp.content:   # aiohttp yields one SSE line at a time
                line = raw.decode("utf-8", "ignore").strip()
                if not line.startswith("data:"):
                    continue
                data = line[5:].strip()
                if data == "[DONE]":
                    break
                chunk = json.loads(data)
                now = time.perf_counter()
                if chunk.get("choices"):
                    # count every token chunk, even if its text is "" (partial UTF-8)
                    if last is None:
                        res.ttft = now - t0
                    else:
                        res.itl.append(now - last)
                    last = now
                    n_chunks += 1
                usage = chunk.get("usage")
                if usage:
                    res.prompt_tokens = usage.get("prompt_tokens", 0)
                    res.output_tokens = usage.get("completion_tokens", 0)
        res.e2e = time.perf_counter() - t0
        if not res.output_tokens:
            res.output_tokens = n_chunks
        res.ok = last is not None
        if not res.ok:
            res.error = "no tokens streamed"
    except Exception as exc:  # noqa: BLE001
        res.error = f"{type(exc).__name__}: {exc}"
        res.e2e = time.perf_counter() - t0
    return res


async def run_load(base_url: str, requests: list[Request], concurrency: int,
                   policy: str = "closed", model: str = "bench",
                   request_timeout_s: int = 1800) -> tuple[list[RequestResult], float]:
    """Fire `requests` at the server with `concurrency` users. Returns (results, wall_s)."""
    url = base_url + "/v1/completions"
    conn = aiohttp.TCPConnector(limit=0)                # no client-side connection cap
    timeout = aiohttp.ClientTimeout(total=request_timeout_s)
    async with aiohttp.ClientSession(connector=conn, timeout=timeout) as session:
        t_level0 = time.perf_counter()
        if policy == "closed":
            sem = asyncio.Semaphore(concurrency)

            async def user(i, r):
                async with sem:
                    return await _send(session, url, model, r, i, t_level0)

            results = await asyncio.gather(*(user(i, r) for i, r in enumerate(requests)))
        elif policy == "waves":
            results = []
            for w in range(0, len(requests), concurrency):
                wave = requests[w:w + concurrency]
                results += await asyncio.gather(
                    *(_send(session, url, model, r, w + j, t_level0) for j, r in enumerate(wave)))
        else:
            raise ValueError(f"unknown policy {policy!r}")
        wall = time.perf_counter() - t_level0
    return list(results), wall


async def warmup(base_url: str, requests: list[Request], model: str = "bench") -> None:
    """A few throwaway requests: first-call costs (CUDA graph replay paths,
    allocator growth, tokenizer threads) shouldn't land in the C=1 numbers."""
    await run_load(base_url, requests, concurrency=len(requests), model=model)

In [ ]:
%%writefile {PROJECT_DIR}/infbench/metrics.py
"""Turn raw per-request timings + monitor samples into one summary row.

Units (per project spec): latencies in milliseconds, throughput in tokens/s.
"""
from __future__ import annotations

import numpy as np

from .loadgen import RequestResult


def _pct(xs, p):
    return float(np.percentile(xs, p)) if len(xs) else float("nan")


def _stats(prefix: str, xs_s: list[float]) -> dict:
    ms = [x * 1000 for x in xs_s]
    return {
        f"{prefix}_mean_ms": float(np.mean(ms)) if ms else float("nan"),
        f"{prefix}_p50_ms": _pct(ms, 50),
        f"{prefix}_p95_ms": _pct(ms, 95),
        f"{prefix}_p99_ms": _pct(ms, 99),
    }


def summarize(results: list[RequestResult], wall_s: float, samples: list[dict],
              counter_delta: dict, extra: dict | None = None) -> dict:
    ok = [r for r in results if r.ok]
    row: dict = dict(extra or {})
    row["n_requests"] = len(results)
    row["n_failed"] = len(results) - len(ok)
    row["duration_s"] = wall_s

    if ok:
        # Throughput over the actual busy window (first send -> last finish)
        t_first = min(r.start for r in ok)
        t_last = max(r.start + r.e2e for r in ok)
        busy = max(t_last - t_first, 1e-9)
        out_tok = sum(r.output_tokens for r in ok)
        in_tok = sum(r.prompt_tokens for r in ok)
        row["req_throughput_rps"] = len(ok) / busy
        row["output_tok_per_s"] = out_tok / busy
        row["total_tok_per_s"] = (out_tok + in_tok) / busy
        row["mean_prompt_tokens"] = in_tok / len(ok)
        row["mean_output_tokens"] = out_tok / len(ok)
        row["output_len_mismatch"] = sum(r.output_tokens != r.expected_output_tokens for r in ok)

        row.update(_stats("ttft", [r.ttft for r in ok]))
        # Steady state: drop the first cohort (all C users send at t=0 and queue
        # behind each other's prefill), which otherwise dominates TTFT p95.
        c = int((extra or {}).get("concurrency") or 0)
        steady = sorted(ok, key=lambda r: r.start)[c:] if c and len(ok) > c else []
        if steady:
            row.update({k.replace("ttft", "ttft_ss"): v for k, v in _stats("ttft", [r.ttft for r in steady]).items()})
            row.update({k.replace("e2e", "e2e_ss"): v for k, v in _stats("e2e", [r.e2e for r in steady]).items()})
        row.update(_stats("itl", [g for r in ok for g in r.itl]))       # pooled token gaps
        row.update(_stats("e2e", [r.e2e for r in ok]))
        tpot = [(r.e2e - r.ttft) / (r.output_tokens - 1) for r in ok if r.output_tokens > 1]
        row.update(_stats("tpot", tpot))                                  # per-request avg
        # What one user "feels": decode speed of their own stream
        row["per_user_tok_per_s"] = float(np.mean([1 / t for t in tpot])) if tpot else float("nan")

    def col(k):
        return [s[k] for s in samples if s.get(k) is not None]

    for k, agg in (("kv_usage", "mean"), ("kv_usage", "max"), ("running", "max"),
                   ("waiting", "max"), ("vram_used_mib", "max"), ("gpu_util_pct", "mean"),
                   ("cpu_pct", "mean"), ("ram_used_mib", "max")):
        v = col(k)
        if v:
            val = float(np.mean(v) if agg == "mean" else np.max(v))
            if k == "kv_usage":
                val *= 100  # 0..1 -> percent
                k = "kv_usage_pct"
            row[f"{k}_{agg}"] = val

    row["preemptions"] = counter_delta.get("preemptions", 0.0)
    q = counter_delta.get("prefix_queries", 0.0)
    row["prefix_hit_rate_pct"] = 100 * counter_delta.get("prefix_hits", 0.0) / q if q else 0.0
    return row

In [ ]:
%%writefile {PROJECT_DIR}/infbench/runner.py
"""Experiment driver: config matrix x concurrency levels -> results on Google Drive.

Layout (one folder per experiment, like systalyze's runs/<name>/):
  <results_dir>/<experiment>/summary.csv              one row per (config, concurrency)
  <results_dir>/<experiment>/<config>/command.txt     exact vllm serve line
  <results_dir>/<experiment>/<config>/server.log
  <results_dir>/<experiment>/<config>/c<C>_requests.jsonl   raw per-request timings
  <results_dir>/<experiment>/<config>/c<C>_monitor.csv      1 Hz GPU/CPU/KV samples

Colab sessions die. The runner is resumable: (config, concurrency) pairs
already present in summary.csv are skipped on the next run.
"""
from __future__ import annotations

import asyncio
import csv
import json
import time
from pathlib import Path

import nest_asyncio
import yaml

from . import workload as wl
from .loadgen import run_load, warmup
from .metrics import summarize
from .monitor import ResourceMonitor
from .server import VLLMServer, config_to_args

nest_asyncio.apply()  # Colab already runs an event loop; let asyncio.run() nest inside it


def load_plan(path: str | Path) -> dict:
    with open(path) as f:
        return yaml.safe_load(f)


def _done(summary_csv: Path) -> set[tuple[str, int]]:
    if not summary_csv.exists():
        return set()
    with open(summary_csv) as f:
        return {(r["config"], int(r["concurrency"])) for r in csv.DictReader(f)}


def _append(summary_csv: Path, row: dict) -> None:
    rows = []
    if summary_csv.exists():
        with open(summary_csv) as f:
            rows = list(csv.DictReader(f))
    rows.append({k: v for k, v in row.items()})
    keys: list[str] = []
    for r in rows:
        keys += [k for k in r if k not in keys]
    with open(summary_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)


def _n_requests(c: int, wcfg: dict, cfg: dict) -> int:
    per_user = cfg.get("requests_per_user", wcfg.get("requests_per_user", 2))
    n = max(cfg.get("min_requests", wcfg.get("min_requests", 20)), per_user * c)
    if cfg.get("max_requests"):
        n = min(n, cfg["max_requests"])
    return max(n, 1)


def _build_requests(kind: str, stream, corpus, n: int, wcfg: dict, exp: dict):
    isl, osl = wcfg["isl"], wcfg["osl"]
    if kind == "fixed":
        return wl.build_fixed(stream, n, isl, osl)
    if kind == "shared_prefix":
        return wl.build_shared_prefix(stream, n, exp["prefix_len"], isl - exp["prefix_len"], osl)
    if kind == "variable_osl":
        return wl.build_variable_osl(stream, corpus, n, isl, scale=exp.get("osl_scale", 0.5),
                                     cap=exp.get("osl_cap", 256), seed=n)
    raise ValueError(kind)


def run_experiment(plan: dict, exp_name: str, results_dir: str | Path,
                   only: list[str] | None = None) -> Path:
    from transformers import AutoTokenizer

    exp = plan["experiments"][exp_name]
    wcfg = plan["workload"]
    out = Path(results_dir) / exp_name
    out.mkdir(parents=True, exist_ok=True)
    summary_csv = out / "summary.csv"
    done = _done(summary_csv)

    tok = AutoTokenizer.from_pretrained(plan.get("tokenizer", plan["model"]))
    corpus = wl.load_corpus(Path(results_dir) / "_cache")
    base_stream = wl.TokenStream(tok, corpus, seed=wcfg.get("seed", 0))

    # Give every concurrency level its own, big-enough region of the token stream
    # (sized by how many prompts that level needs), so levels never share prompts.
    all_levels = sorted(set(exp["levels"]) | {l for cf in exp["configs"] for l in cf.get("levels", [])})
    isl_max = max([wcfg["isl"]] + [cf.get("isl", 0) for cf in exp["configs"]])
    offsets, cur = {}, 0
    for c in all_levels:
        offsets[c] = cur
        cur += max(wcfg.get("min_requests", 20), wcfg.get("requests_per_user", 4) * c) * isl_max
    if cur > len(base_stream.ids) - 4096:
        print(f"[runner] note: experiment needs {cur:,} prompt tokens, corpus has "
              f"{len(base_stream.ids):,}; some prompts repeat (harmless unless prefix caching is on)")

    server: VLLMServer | None = None
    server_key = None
    try:
        for cfg_over in exp["configs"]:
            cfg = {**plan.get("defaults", {}), **cfg_over}
            name = cfg["name"]
            if cfg.get("enabled") is False or (only and name not in only):
                continue
            levels = cfg.get("levels", exp["levels"])
            todo = [c for c in levels if (name, c) not in done]
            if not todo:
                print(f"[runner] {exp_name}/{name}: all levels done, skipping")
                continue

            model = cfg.get("model", plan["model"])
            args = config_to_args(cfg)
            cdir = out / name
            cdir.mkdir(exist_ok=True)

            engine = cfg.get("engine", "vllm")
            key = (engine, model, tuple(args))
            if key != server_key:              # reuse the server if nothing changed
                if server:
                    server.stop()
                server = VLLMServer(model, args, cdir / "server.log", port=plan.get("port", 8000),
                                    engine=engine)
                (cdir / "command.txt").write_text(" ".join(server.command) + "\n")
                server.start(timeout_s=plan.get("server_timeout_s", 900))
                server_key = key
                # warm-up prompts come from the END of the stream so they never
                # overlap (and pre-cache) any measured prompt
                n_wu = wcfg.get("warmup_requests", 4)
                wu = base_stream.view(len(base_stream.ids) - n_wu * wcfg["isl"] - 1)
                asyncio.run(warmup(server.base_url, wl.build_fixed(wu, n_wu, wcfg["isl"], 16)))
            kv_cap = server.kv_capacity()

            # a config may override the prompt/output length (prompt-length sweep)
            wc = {**wcfg, **{k: cfg[k] for k in ("isl", "osl") if k in cfg}}
            for c in todo:
                # Every config replays the SAME prompts at a given level -> apples-to-apples
                stream = base_stream.view(offsets[c])
                n = _n_requests(c, wcfg, cfg)
                reqs = _build_requests(exp.get("workload", "fixed"), stream, corpus, n, wc, exp)
                policy = cfg.get("policy", exp.get("policy", "closed"))
                print(f"[runner] {exp_name}/{name}  C={c}  n={n}  policy={policy}")

                async def _level():
                    async with ResourceMonitor(server.base_url, plan.get("monitor_interval_s", 1.0)) as mon:
                        res, wall = await run_load(server.base_url, reqs, c, policy=policy)
                    return res, wall, mon

                res, wall, mon = asyncio.run(_level())
                row = summarize(res, wall, mon.samples, mon.counter_delta, extra={
                    "experiment": exp_name, "config": name, "concurrency": c, "policy": policy,
                    "engine": engine,
                    "model": model, "isl": wc["isl"],
                    "osl": wc["osl"] if exp.get("workload") != "variable_osl" else "variable",
                    "max_num_seqs": cfg.get("max_num_seqs"),
                    "max_num_batched_tokens": cfg.get("max_num_batched_tokens") or "default",
                    "gpu_memory_utilization": cfg.get("gpu_memory_utilization"),
                    "quantization": cfg.get("quantization") or "none",
                    "prefix_caching": bool(cfg.get("prefix_caching")),
                    "kv_cache_dtype": cfg.get("kv_cache_dtype") or "auto",
                    **kv_cap, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                })
                with open(cdir / f"c{c}_requests.jsonl", "w") as f:
                    for r in res:
                        f.write(json.dumps(r.to_row()) + "\n")
                if mon.samples:
                    keys = sorted({k for s in mon.samples for k in s})
                    with open(cdir / f"c{c}_monitor.csv", "w", newline="") as f:
                        w = csv.DictWriter(f, fieldnames=keys)
                        w.writeheader()
                        w.writerows(mon.samples)
                _append(summary_csv, row)
                _print_row(row)
                if row["n_failed"]:
                    first_err = next(r.error for r in res if not r.ok)
                    print(f"   !! {row['n_failed']} failed requests, e.g. {first_err}")
    finally:
        if server:
            server.stop()
    return summary_csv


def _print_row(r: dict) -> None:
    print(f"   TTFT p95 {r.get('ttft_p95_ms', float('nan')):8.1f} ms | "
          f"ITL p95 {r.get('itl_p95_ms', float('nan')):6.1f} ms | "
          f"E2E p95 {r.get('e2e_p95_ms', float('nan')):8.1f} ms | "
          f"{r.get('output_tok_per_s', float('nan')):7.1f} out tok/s | "
          f"KV max {r.get('kv_usage_pct_max', float('nan')):5.1f}% | "
          f"preempt {r.get('preemptions', 0):.0f}")

In [ ]:
%%writefile {PROJECT_DIR}/infbench/analyze.py
"""Plots + comparison tables from summary.csv files (matplotlib, Colab-friendly)."""
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Fixed categorical order; a config keeps its color in every figure.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "lines.linewidth": 2, "font.size": 10,
})


def load(results_dir: str | Path, experiment: str) -> pd.DataFrame:
    df = pd.read_csv(Path(results_dir) / experiment / "summary.csv")
    return df.sort_values(["config", "concurrency"])


def _style(configs):
    return {c: (PALETTE[i % 8], MARKERS[i % 8]) for i, c in enumerate(configs)}


def sweep_grid(df: pd.DataFrame, panels=None, title: str = "", save: str | Path | None = None):
    """Small multiples: one panel per metric, x = concurrent users, one line per config."""
    ttft_col = "ttft_ss_p95_ms" if "ttft_ss_p95_ms" in df and df["ttft_ss_p95_ms"].notna().any() else "ttft_p95_ms"
    panels = panels or [
        (ttft_col, "TTFT p95, steady state (ms)" if ttft_col.startswith("ttft_ss") else "TTFT p95 (ms)", True),
        ("itl_p50_ms", "Inter-token latency p50 (ms)", False),
        ("e2e_p95_ms", "End-to-end latency p95 (ms)", True),
        ("output_tok_per_s", "Output throughput (tok/s)", False),
        ("kv_usage_pct_max", "Peak KV-cache usage (%)", False),
        ("vram_used_mib_max", "Peak VRAM used (MiB)", False),
    ]
    configs = list(dict.fromkeys(df["config"]))
    sty = _style(configs)
    ncol = 3
    nrow = -(-len(panels) // ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 3.6 * nrow))
    for ax, (col, label, logy) in zip(axes.flat, panels):
        if col not in df:
            ax.set_visible(False)
            continue
        dashed_any = False
        for cfg in configs:
            d = df[df.config == cfg]
            color, m = sty[cfg]
            y, ls = d[col], "-"
            if col.startswith("ttft_ss") and d[col].iloc[1:].isna().all():
                # n == C at every level: no steady state exists, show all-request p95 dashed
                y, ls, dashed_any = d["ttft_p95_ms"], "--", True
            ax.plot(d.concurrency, y, marker=m, markersize=6, color=color, label=cfg, linestyle=ls)
        if dashed_any:
            label += "\n(dashed: all requests, no steady state)"
        ax.set_title(label, loc="left", fontsize=10)
        ax.set_xlabel("Concurrent users")
        ax.set_xscale("symlog", linthresh=10)
        ax.set_xticks(sorted(df.concurrency.unique()))
        ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
        if logy:
            ax.set_yscale("log")
    for ax in list(axes.flat)[len(panels):]:
        ax.set_visible(False)
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=min(len(configs), 5), frameon=False,
               bbox_to_anchor=(0.5, 1.02))
    if title:
        fig.suptitle(title, y=1.07, fontsize=12)
    fig.tight_layout()
    if save:
        fig.savefig(save, bbox_inches="tight")
    return fig


def compare_table(df: pd.DataFrame, baseline: str, metrics=None) -> pd.DataFrame:
    """Each config vs baseline at every concurrency level, as a multiplier.
    Latency: baseline / config (>1 = faster). Throughput: config / baseline."""
    metrics = metrics or {"ttft_p95_ms": "lower", "e2e_p95_ms": "lower",
                          "itl_p95_ms": "lower", "output_tok_per_s": "higher"}
    base = df[df.config == baseline].set_index("concurrency")
    rows = []
    for cfg, d in df[df.config != baseline].groupby("config"):
        d = d.set_index("concurrency")
        for c in d.index.intersection(base.index):
            r = {"config": cfg, "concurrency": c}
            for m, better in metrics.items():
                if m in d and m in base:
                    r[f"{m} x"] = (base.at[c, m] / d.at[c, m]) if better == "lower" \
                        else (d.at[c, m] / base.at[c, m])
            rows.append(r)
    return pd.DataFrame(rows).round(2)


def headline_table(df: pd.DataFrame) -> pd.DataFrame:
    cols = ["config", "concurrency", "ttft_p50_ms", "ttft_p95_ms", "ttft_ss_p95_ms", "itl_p50_ms",
            "itl_p95_ms", "e2e_p95_ms", "e2e_ss_p95_ms",
            "e2e_p99_ms", "output_tok_per_s", "req_throughput_rps", "per_user_tok_per_s",
            "kv_usage_pct_max", "waiting_max", "preemptions", "prefix_hit_rate_pct",
            "vram_used_mib_max", "gpu_util_pct_mean", "cpu_pct_mean", "n_failed"]
    return df[[c for c in cols if c in df]].round(1)


def timeline(results_dir, experiment, config, concurrency, save=None):
    """KV-cache usage and queue depth over one run - shows *when* the KV cache fills
    and requests start waiting. Two panels, shared x (never a dual y-axis)."""
    p = Path(results_dir) / experiment / config / f"c{concurrency}_monitor.csv"
    m = pd.read_csv(p)
    fig, (a, b) = plt.subplots(2, 1, figsize=(8, 4.5), sharex=True)
    a.plot(m.t, m.kv_usage * 100, color=PALETTE[0])
    a.set_title(f"KV-cache usage (%) - {config}, C={concurrency}", loc="left", fontsize=10)
    a.set_ylim(0, 105)
    b.plot(m.t, m.running, color=PALETTE[0], label="running")
    b.plot(m.t, m.waiting, color=PALETTE[1], label="waiting")
    b.set_title("Requests running vs waiting", loc="left", fontsize=10)
    b.set_xlabel("Seconds since level start")
    b.legend(frameon=False)
    fig.tight_layout()
    if save:
        fig.savefig(save, bbox_inches="tight")
    return fig


def bar_compare(df: pd.DataFrame, metric: str, label: str, save=None):
    """Grouped bars: configs side by side at each concurrency (for 2-config experiments)."""
    configs = list(dict.fromkeys(df["config"]))
    levels = sorted(df.concurrency.unique())
    w = 0.8 / len(configs)
    fig, ax = plt.subplots(figsize=(1.8 * len(levels) + 2, 3.4))
    for i, cfg in enumerate(configs):
        d = df[df.config == cfg].set_index("concurrency").reindex(levels)
        xs = [k + (i - (len(configs) - 1) / 2) * w for k in range(len(levels))]
        ax.bar(xs, d[metric], width=w * 0.92, color=PALETTE[i % 8], label=cfg)
    ax.set_xticks(range(len(levels)), [str(c) for c in levels])
    ax.set_xlabel("Concurrent users")
    ax.set_title(label, loc="left", fontsize=10)
    ax.legend(frameon=False)
    fig.tight_layout()
    if save:
        fig.savefig(save, bbox_inches="tight")
    return fig


def itl_histogram(results_dir, experiment, config, levels=(25, 50, 100), save=None):
    """Distribution of token gaps. A bimodal shape (most gaps ~tens of ms plus a
    cluster at ~400 ms) is the signature of generation stalls: decode steps that
    also had to process newcomers' prompts."""
    import json
    import numpy as np
    fig, ax = plt.subplots(figsize=(8, 3.6))
    bins = np.logspace(np.log10(3), np.log10(3000), 70)
    for i, c in enumerate(levels):
        p = Path(results_dir) / experiment / config / f"c{c}_requests.jsonl"
        if not p.exists():
            continue
        g = np.array([x for line in open(p) for x in json.loads(line)["itl"]]) * 1000
        ax.hist(g, bins=bins, histtype="step", linewidth=2, color=PALETTE[i % 8],
                label=f"C={c}  ({100 * (g > 200).mean():.1f}% of gaps > 200 ms)")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Gap between consecutive tokens (ms, log scale)")
    ax.set_title(f"Inter-token gap distribution - {config}", loc="left", fontsize=10)
    ax.legend(frameon=False)
    fig.tight_layout()
    if save:
        fig.savefig(save, bbox_inches="tight")
    return fig


def metric_vs(df: pd.DataFrame, x: str, y: str, group: str, label: str, xlabel: str,
              logx: bool = False, save=None):
    """One line per group (e.g. engine) of metric y against a numeric x (e.g. isl)."""
    fig, ax = plt.subplots(figsize=(6, 3.6))
    for i, (g, d) in enumerate(df.sort_values(x).groupby(group, sort=False)):
        ax.plot(d[x], d[y], marker=MARKERS[i % 8], color=PALETTE[i % 8], label=str(g))
    if logx:
        ax.set_xscale("log", base=2)
        ax.set_xticks(sorted(df[x].unique()))
        ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.set_xlabel(xlabel)
    ax.set_title(label, loc="left", fontsize=10)
    ax.legend(frameon=False)
    fig.tight_layout()
    if save:
        fig.savefig(save, bbox_inches="tight")
    return fig


def steady_state(results_dir, experiment, df: pd.DataFrame | None = None) -> pd.DataFrame:
    """Recompute steady-state TTFT/E2E from raw per-request files, excluding the first
    C requests of each level (the t=0 burst cohort). Works on results collected
    before these columns existed in summary.csv."""
    import json
    import numpy as np
    df = load(results_dir, experiment) if df is None else df.copy()
    for i, r in df.iterrows():
        p = Path(results_dir) / experiment / r["config"] / f"c{int(r['concurrency'])}_requests.jsonl"
        if not p.exists():
            continue
        reqs = sorted((json.loads(l) for l in open(p)), key=lambda x: x["start"])
        steady = [q for q in reqs[int(r["concurrency"]):] if q.get("ok")]
        if not steady:
            continue
        for key, col in (("ttft", "ttft_ss"), ("e2e", "e2e_ss")):
            v = np.array([q[key] for q in steady]) * 1000
            df.loc[i, f"{col}_p50_ms"] = np.percentile(v, 50)
            df.loc[i, f"{col}_p95_ms"] = np.percentile(v, 95)
    return df

In [ ]:
%%writefile {PROJECT_DIR}/infbench/kv_ablation.py
"""What does the KV cache actually buy?  (single request, Hugging Face transformers)

vLLM can't turn the KV cache off (PagedAttention *is* a KV cache), so this
micro-benchmark uses a hand-written greedy decode loop in transformers:

  with cache    : each step feeds ONE new token and reuses stored K/V
                  -> per-step cost ~ constant (memory-bound weight reads)
  without cache : each step re-runs attention over the WHOLE sequence
                  -> per-step cost grows with sequence length (O(n^2) total)

Run it while no vLLM server is holding the GPU.
"""
from __future__ import annotations

import time

import torch


@torch.inference_mode()
def decode_step_times(model, input_ids: torch.Tensor, new_tokens: int, use_cache: bool) -> list[float]:
    times = []
    ids = input_ids
    past = None
    nxt = None
    for step in range(new_tokens):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        if use_cache:
            out = model(ids if past is None else nxt, past_key_values=past, use_cache=True)
            past = out.past_key_values
        else:
            out = model(ids, use_cache=False)
        nxt = out.logits[:, -1:].argmax(-1)
        ids = torch.cat([ids, nxt], dim=1)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return times


def run(model_name: str = "Qwen/Qwen2.5-1.5B-Instruct", prompt_lens=(128, 512, 1024),
        new_tokens: int = 128, prompt_ids: list[int] | None = None):
    import pandas as pd
    from transformers import AutoModelForCausalLM

    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16).cuda().eval()
    rows, curves = [], {}
    for L in prompt_lens:
        ids = torch.tensor([prompt_ids[:L]] if prompt_ids else
                           torch.randint(100, 20000, (1, L)).tolist(), device="cuda")
        decode_step_times(model, ids, 4, True)  # warm-up
        for cache in (True, False):
            torch.cuda.reset_peak_memory_stats()
            t = decode_step_times(model, ids, new_tokens, cache)
            curves[(L, cache)] = t
            decode = t[1:]  # step 0 is the prefill
            rows.append({
                "prompt_len": L, "kv_cache": cache,
                "ttft_ms": t[0] * 1000,
                "mean_step_ms": 1000 * sum(decode) / len(decode),
                "last_step_ms": 1000 * decode[-1],
                "decode_tok_per_s": len(decode) / sum(decode),
                "peak_mem_mib": torch.cuda.max_memory_allocated() / 2**20,
            })
    del model
    torch.cuda.empty_cache()
    return pd.DataFrame(rows).round(2), curves


def plot(curves, save=None):
    import matplotlib.pyplot as plt
    from .analyze import PALETTE

    lens = sorted({L for L, _ in curves})
    fig, ax = plt.subplots(figsize=(8, 3.6))
    for i, L in enumerate(lens):
        c = PALETTE[i % 8]
        ax.plot([x * 1000 for x in curves[(L, True)][1:]], color=c, label=f"prompt {L} - KV cache")
        ax.plot([x * 1000 for x in curves[(L, False)][1:]], color=c, linestyle="--",
                label=f"prompt {L} - no cache")
    ax.set_xlabel("Generated token index")
    ax.set_title("Per-token decode latency (ms)", loc="left", fontsize=10)
    ax.legend(frameon=False, ncol=2, fontsize=8)
    fig.tight_layout()
    if save:
        fig.savefig(save, bbox_inches="tight")
    return fig

In [ ]:
%%writefile {PROJECT_DIR}/infbench/hf_server.py
"""Naive Hugging Face transformers server - the "how most LLM apps start" baseline.

This is how a typical first-version chatbot backend serves a model: load it with
transformers, put an HTTP endpoint in front, and run generation for ONE request at
a time (a global lock). It does use a KV cache (model forward with past_key_values),
but has none of the serving-engine tricks:

  * no batching       - other users wait in the lock's queue
  * contiguous KV     - each request's cache grows by torch.cat-style reallocation,
                        no paging, no sharing
  * eager PyTorch     - one Python-driven forward per token, no CUDA graphs

Speaks the same streaming /v1/completions protocol as vLLM (token-id prompts,
max_tokens, min_tokens, usage chunk), so infbench's load generator and monitor
work unchanged. Standalone on purpose: launched as a subprocess by server.py.

    python hf_server.py --model Qwen/Qwen2.5-1.5B-Instruct --port 8000
"""
from __future__ import annotations

import argparse
import asyncio
import json

import torch
from aiohttp import web
from transformers import AutoModelForCausalLM, AutoTokenizer


def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--host", default="127.0.0.1")
    p.add_argument("--port", type=int, default=8000)
    p.add_argument("--served-model-name", default="bench")
    p.add_argument("--max-model-len", type=int, default=2048)
    p.add_argument("--dtype", default="half", choices=["half", "float32"])
    p.add_argument("--device", default="cuda")
    a = p.parse_args()

    dtype = torch.float16 if a.dtype == "half" else torch.float32
    print(f"[hf_server] loading {a.model} ({a.dtype}) on {a.device}", flush=True)
    tok = AutoTokenizer.from_pretrained(a.model)
    model = AutoModelForCausalLM.from_pretrained(a.model, dtype=dtype).to(a.device).eval()
    eos = model.generation_config.eos_token_id
    eos_ids = set(eos if isinstance(eos, list) else [eos]) - {None}
    eos_list = sorted(eos_ids)
    if a.device == "cuda":
        print(f"[hf_server] weights: {torch.cuda.memory_allocated() / 2**20:.0f} MiB", flush=True)

    lock = asyncio.Lock()   # <- the whole point: one generation at a time
    st = {"running": 0, "waiting": 0, "prompt": 0, "gen": 0}

    @torch.inference_mode()
    def step(ids, past, block_eos: bool):
        """One forward pass + greedy pick. Returns (next-token tensor, its id, new cache)."""
        out = model(input_ids=ids, past_key_values=past, use_cache=True)
        logits = out.logits[:, -1, :]
        if block_eos and eos_list:          # min_tokens: no EOS before the target length
            logits[:, eos_list] = float("-inf")
        nxt = logits.argmax(-1, keepdim=True)
        return nxt, int(nxt.item()), out.past_key_values

    def sse(obj) -> bytes:
        return f"data: {json.dumps(obj)}\n\n".encode()

    async def completions(req: web.Request):
        body = await req.json()
        prompt = body["prompt"]
        if isinstance(prompt, str):
            prompt = tok.encode(prompt, add_special_tokens=False)
        max_t = int(body.get("max_tokens") or 16)
        min_t = int(body.get("min_tokens") or 0)
        if len(prompt) + max_t > a.max_model_len:
            return web.json_response({"error": "prompt + max_tokens exceeds max_model_len"}, status=400)

        resp = web.StreamResponse(headers={"Content-Type": "text/event-stream"})
        await resp.prepare(req)
        st["waiting"] += 1
        async with lock:
            st["waiting"] -= 1
            st["running"] += 1
            n = 0
            try:
                ids = torch.tensor([prompt], device=a.device)
                past, nxt = None, None
                while n < max_t:
                    # run the GPU work in a thread so the event loop keeps
                    # accepting (and queueing) other users' requests
                    nxt, t, past = await asyncio.to_thread(
                        step, ids if past is None else nxt, past, n < min_t)
                    n += 1
                    st["gen"] += 1
                    await resp.write(sse({"choices": [{"index": 0, "text": tok.decode([t]),
                                                       "finish_reason": None}]}))
                    if t in eos_ids and n >= min_t:
                        break
            finally:
                st["running"] -= 1
                st["prompt"] += len(prompt)
                del past
        await resp.write(sse({"choices": [], "usage": {
            "prompt_tokens": len(prompt), "completion_tokens": n,
            "total_tokens": len(prompt) + n}}))
        await resp.write(b"data: [DONE]\n\n")
        return resp

    async def health(_):
        return web.Response(text="ok")

    async def metrics(_):
        return web.Response(text=(
            f"infbench:num_requests_running {st['running']}\n"
            f"infbench:num_requests_waiting {st['waiting']}\n"
            f"infbench:prompt_tokens_total {st['prompt']}\n"
            f"infbench:generation_tokens_total {st['gen']}\n"))

    app = web.Application(client_max_size=64 * 2**20)
    app.router.add_get("/health", health)
    app.router.add_get("/metrics", metrics)
    app.router.add_post("/v1/completions", completions)
    print("[hf_server] ready", flush=True)
    web.run_app(app, host=a.host, port=a.port, print=None)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {PROJECT_DIR}/configs/experiments.yaml
# Experiment plan for a Colab free-tier T4 (15 GB, compute capability 7.5).
# One `defaults` block + per-config overrides, the same pattern as
# systalyze's row.json (deployment as data) and the studio's combos.yaml.

model: Qwen/Qwen2.5-1.5B-Instruct
tokenizer: Qwen/Qwen2.5-1.5B-Instruct   # AWQ/GPTQ variants share this vocab
port: 8000
server_timeout_s: 900
monitor_interval_s: 1.0

defaults:
  dtype: half                  # T4 has no bfloat16
  max_model_len: 2048
  gpu_memory_utilization: 0.85
  max_num_seqs: 256            # vLLM's default batch ceiling = continuous batching ON
  prefix_caching: false        # off unless an experiment is about it

workload:
  isl: 512                     # EXACT input tokens for every request
  osl: 128                     # EXACT output tokens (min_tokens == max_tokens)
  requests_per_user: 4         # n_requests = max(min_requests, 4 * C); more samples -> steadier p95/p99
  min_requests: 20
  warmup_requests: 4
  seed: 0

experiments:

  # 1) Main result: baseline vs optimizations across concurrency
  concurrency_sweep:
    workload: fixed
    policy: closed
    levels: [1, 10, 25, 50, 100]
    configs:
      - name: baseline_hf_naive         # plain transformers + HTTP, one request at a time:
        engine: hf                      # how a first-version chatbot backend usually serves
        requests_per_user: 1            # serial and slow (~15 min for all levels)
        min_requests: 10
      - name: vllm_no_batching          # vLLM engine, but one sequence at a time
        max_num_seqs: 1
        requests_per_user: 1            # serial serving is slow; keep the run short
        min_requests: 10
      - name: cb_fp16                   # continuous batching + paged KV cache
      - name: cb_fp16_small_kv          # same, but starve the KV cache -> queueing / preemption
        gpu_memory_utilization: 0.35
      - name: cb_awq_int4               # 4-bit weights
        model: Qwen/Qwen2.5-1.5B-Instruct-AWQ
        quantization: awq
      - name: cb_gptq_int8              # 8-bit weights (optional, ~5 extra minutes)
        model: Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int8
        quantization: gptq
        enabled: false
      - name: cb_fp16_kv_fp8            # FP8 KV cache: 2x KV capacity. Experimental on T4.
        kv_cache_dtype: fp8
        enabled: false

  # 2) KV-cache reuse: automatic prefix caching on a shared-system-prompt workload
  prefix_caching:
    workload: shared_prefix
    prefix_len: 448                   # shared tokens; 64 unique tokens per request
    policy: closed
    levels: [1, 10, 50]
    configs:
      - name: prefix_cache_off
        prefix_caching: false
      - name: prefix_cache_on
        prefix_caching: true

  # 3) Why continuous batching matters: variable output lengths, static vs continuous
  batching_policy:
    workload: variable_osl            # output lengths drawn from the systalyze workload
    osl_scale: 0.5                    # p50 ~100, p90 ~200 tokens
    osl_cap: 256
    levels: [10, 50]
    configs:
      - name: continuous_batching
        policy: closed
      - name: static_batching_waves
        policy: waves

  # 5) Generation stalls: how big a step may get when new prompts join the batch.
  #    vLLM's chunked prefill packs up to max_num_batched_tokens into ONE step
  #    (2048 by default on this setup). Smaller budget -> shorter decode stalls,
  #    but prompts take more steps -> higher TTFT. Trade-off: TTFT vs ITL tail.
  chunked_prefill:
    workload: fixed
    policy: closed
    levels: [50, 100]
    configs:
      - name: mnbt_512
        max_num_batched_tokens: 512
      - name: mnbt_1024
        max_num_batched_tokens: 1024
      - name: mnbt_2048            # what vLLM picked by default in experiment 1
        max_num_batched_tokens: 2048
      - name: mnbt_4096
        max_num_batched_tokens: 4096

  # 6) Prefill cost vs prompt length (output fixed at 128). One server per engine.
  #    If vLLM's TTFT gap vs HF grows with length -> attention kernel (TRITON_ATTN on sm75);
  #    if the gap is constant -> fixed per-request overhead.
  prompt_length:
    workload: fixed
    policy: closed
    levels: [1, 25]
    configs:
      - {name: vllm_isl_128,  isl: 128}
      - {name: vllm_isl_256,  isl: 256}
      - {name: vllm_isl_512,  isl: 512}
      - {name: vllm_isl_1024, isl: 1024}
      - {name: hf_isl_128,  isl: 128,  engine: hf, levels: [1], min_requests: 10}
      - {name: hf_isl_256,  isl: 256,  engine: hf, levels: [1], min_requests: 10}
      - {name: hf_isl_512,  isl: 512,  engine: hf, levels: [1], min_requests: 10}
      - {name: hf_isl_1024, isl: 1024, engine: hf, levels: [1], min_requests: 10}

  # 7) Model size: decode is memory-bound, so single-user ITL should scale with
  #    weight bytes - until fixed overheads set a floor (watch the 0.5B model).
  model_size:
    workload: fixed
    policy: closed
    levels: [1, 25, 100]
    configs:
      - name: qwen2.5_0.5b
        model: Qwen/Qwen2.5-0.5B-Instruct
      - name: qwen2.5_1.5b
      - name: qwen2.5_3b
        model: Qwen/Qwen2.5-3B-Instruct

  # 8) Admission control: cap how many requests share the GPU at once.
  #    Lower cap -> each admitted user decodes faster, but more users queue.
  admission_control:
    workload: fixed
    policy: closed
    levels: [100]
    configs:
      - {name: max_seqs_16,  max_num_seqs: 16}
      - {name: max_seqs_32,  max_num_seqs: 32}
      - {name: max_seqs_64,  max_num_seqs: 64}
      - {name: max_seqs_256, max_num_seqs: 256}

## 5 · Import and load the plan

In [ ]:
import sys, importlib, copy, asyncio
import nest_asyncio
nest_asyncio.apply()          # Colab already runs an event loop
sys.path.insert(0, PROJECT_DIR)

import infbench.runner as runner, infbench.analyze as analyze
importlib.reload(runner); importlib.reload(analyze)

plan = runner.load_plan(f'{PROJECT_DIR}/configs/experiments.yaml')
for name, exp in plan['experiments'].items():
    print(f"{name:20s} levels={exp['levels']}  configs={[c['name'] for c in exp['configs'] if c.get('enabled', True)]}")

## 6 · Smoke test (~3 min)
One config, two concurrency levels, into a separate `smoke/` folder. Confirms vLLM boots on the
T4, streaming works, `/metrics` is scraped, and every request got exactly 128 output tokens
(`output_len_mismatch` must be 0).

In [ ]:
smoke = copy.deepcopy(plan)
smoke['experiments'] = {'smoke': {**plan['experiments']['concurrency_sweep'], 'levels': [1, 10],
                                  'configs': [{'name': 'cb_fp16'}]}}
csv_path = runner.run_experiment(smoke, 'smoke', RESULTS_DIR)
df = analyze.load(RESULTS_DIR, 'smoke')
df[['config','concurrency','ttft_p95_ms','itl_p95_ms','e2e_p95_ms','output_tok_per_s',
    'kv_usage_pct_max','vram_used_mib_max','kv_capacity_tokens','output_len_mismatch','n_failed']]

## 7 · Experiment 1 — concurrency sweep: baseline vs optimizations (~50–60 min)

| config | what changes | the question it answers |
|---|---|---|
| `baseline_hf_naive` | plain `transformers` + HTTP, global lock | How does a typical first-version chatbot backend behave when users overlap? |
| `vllm_no_batching` | vLLM, `--max-num-seqs 1` | How much comes from a better **engine** alone (CUDA graphs, fused kernels), with batching still off? |
| `cb_fp16` | default (256 seqs) | How much does **continuous batching + paged KV cache** help? |
| `cb_fp16_small_kv` | `--gpu-memory-utilization 0.35` | What happens when the **KV cache runs out** (queueing, preemption)? |
| `cb_awq_int4` | AWQ 4-bit weights | Does **quantization** help speed, memory, or both — and at which load? |
| `cb_gptq_int8`, `cb_fp16_kv_fp8` | disabled by default | Optional extras: set `enabled: true` in the YAML |

In [ ]:
runner.run_experiment(plan, 'concurrency_sweep', RESULTS_DIR)

## 8 · Experiment 2 — KV-cache reuse: automatic prefix caching (~8 min)
Every request shares a 448-token prefix (think: system prompt or RAG context) + 64 unique tokens.
With prefix caching on, vLLM reuses the KV blocks of the shared prefix instead of recomputing them,
so prefill shrinks from 512 to ~64 tokens per request. Watch **TTFT** and `prefix_hit_rate_pct`.

In [ ]:
runner.run_experiment(plan, 'prefix_caching', RESULTS_DIR)

## 9 · Experiment 3 — static vs continuous batching with variable output lengths (~8 min)
With fixed output lengths every request finishes together, so static and continuous batching look
alike. Real traffic has variable lengths (here sampled from the systalyze distribution). In
**static waves**, a slot freed by a short answer sits idle until the longest answer in the wave
finishes; continuous batching refills it immediately.

In [ ]:
runner.run_experiment(plan, 'batching_policy', RESULTS_DIR)

## 10 · Experiment 4 — what the KV cache itself buys (Hugging Face, single request, ~5 min)
vLLM can't switch its KV cache off, so this uses a hand-written decode loop in `transformers`.
Without a cache, every new token re-processes the whole sequence, so per-token latency grows with
length; with a cache it stays flat. Runs only when no vLLM server holds the GPU.

In [ ]:
import infbench.kv_ablation as kv
kv_df, curves = kv.run(plan['model'], prompt_lens=(128, 512, 1024), new_tokens=128)
kv_df.to_csv(f'{RESULTS_DIR}/kv_ablation.csv', index=False)
kv.plot(curves, save=f'{RESULTS_DIR}/kv_ablation.png')
kv_df

## 11 · Experiment 5 — generation stalls vs chunked-prefill budget (~15 min)
Experiment 1 showed most token gaps at 30–70 ms but a cluster at ~430 ms (16% of gaps at C=100).
Those are **decode steps that also processed newcomers' prompts**: vLLM packs up to
`max_num_batched_tokens` (2048 here, ≈ four 512-token prompts) into one step, and everyone already
generating waits for it. A smaller budget shortens each stall but spreads prompts over more steps.
Prediction: ITL p95/p99 falls with the budget while TTFT rises — a direct TTFT ↔ smoothness trade-off.

In [ ]:
runner.run_experiment(plan, 'chunked_prefill', RESULTS_DIR)

## 12 · Experiment 6 — prefill cost vs prompt length, vLLM vs HF (~12 min)
At one user, HF's TTFT (74 ms) beat vLLM's (~125 ms) for a 512-token prompt. vLLM on the T4 uses the
`TRITON_ATTN` backend (FlashAttention needs sm80+). If vLLM's gap grows with prompt length, the
attention kernel is the cause; if it stays constant, it's fixed per-request overhead (API, scheduler).

In [ ]:
runner.run_experiment(plan, 'prompt_length', RESULTS_DIR)

## 13 · Experiment 7 — model size: 0.5B vs 1.5B vs 3B (~20 min)
Decode reads every weight once per step, so single-user ITL should grow with weight bytes. The 0.5B
model tests where fixed overheads (kernel launches, scheduling) set a floor that bandwidth can't explain.

In [ ]:
runner.run_experiment(plan, 'model_size', RESULTS_DIR)

## 14 · Experiment 8 — admission control: max concurrent sequences at C=100 (~12 min)
Capping `max_num_seqs` keeps the batch small: admitted users decode faster, the rest queue.
Look at the per-user speed (`per_user_tok_per_s`) vs TTFT vs total throughput.

In [ ]:
runner.run_experiment(plan, 'admission_control', RESULTS_DIR)

## 15 · Analysis

In [ ]:
sweep = analyze.steady_state(RESULTS_DIR, 'concurrency_sweep')   # adds steady-state TTFT/E2E columns
analyze.sweep_grid(sweep, title='Qwen2.5-1.5B on T4 — 512 in / 128 out tokens',
                   save=f'{RESULTS_DIR}/sweep_grid.png')
analyze.headline_table(sweep)

In [ ]:
# Multipliers vs the naive HF baseline (>1 = better)
analyze.compare_table(sweep, baseline='baseline_hf_naive')

In [ ]:
# Quantization in isolation: AWQ vs FP16, both with continuous batching
analyze.compare_table(sweep[sweep.config.isin(['cb_fp16', 'cb_awq_int4'])], baseline='cb_fp16')

In [ ]:
# When does the KV cache fill up? (look for 'waiting' > 0 and preemptions)
analyze.timeline(RESULTS_DIR, 'concurrency_sweep', 'cb_fp16_small_kv', 100,
                 save=f'{RESULTS_DIR}/timeline_small_kv_c100.png');
sweep[['config','concurrency','kv_capacity_tokens','kv_usage_pct_max','waiting_max','preemptions']]

In [ ]:
pc = analyze.load(RESULTS_DIR, 'prefix_caching')
analyze.bar_compare(pc, 'ttft_p95_ms', 'TTFT p95 (ms) — shared 448-token prefix',
                    save=f'{RESULTS_DIR}/prefix_ttft.png')
pc[['config','concurrency','ttft_p50_ms','ttft_p95_ms','e2e_p95_ms','output_tok_per_s','prefix_hit_rate_pct']].round(1)

In [ ]:
bp = analyze.load(RESULTS_DIR, 'batching_policy')
analyze.bar_compare(bp, 'output_tok_per_s', 'Output throughput (tok/s) — variable output lengths',
                    save=f'{RESULTS_DIR}/batching_policy_throughput.png')
bp[['config','concurrency','output_tok_per_s','e2e_p95_ms','e2e_p99_ms','ttft_p95_ms']].round(1)

In [ ]:
# Generation stalls: bimodal token-gap distribution (experiment 1)
analyze.itl_histogram(RESULTS_DIR, 'concurrency_sweep', 'cb_fp16', levels=(25, 50, 100),
                      save=f'{RESULTS_DIR}/itl_histogram_cb_fp16.png');

In [ ]:
# Experiment 5: chunked-prefill budget -> TTFT vs ITL tail
cp = analyze.load(RESULTS_DIR, 'chunked_prefill')
cp['budget'] = cp.config.str.replace('mnbt_', '').astype(int)
for c in (50, 100):
    d = cp[cp.concurrency == c]
    analyze.metric_vs(d.melt(id_vars='budget', value_vars=['itl_p99_ms', 'ttft_p95_ms']),
                      'budget', 'value', 'variable', f'C={c}: ITL p99 vs TTFT p95 (ms)',
                      'max_num_batched_tokens', logx=True, save=f'{RESULTS_DIR}/chunked_prefill_c{c}.png')
cp[['config','concurrency','ttft_p50_ms','ttft_p95_ms','itl_p50_ms','itl_p95_ms','itl_p99_ms','output_tok_per_s']].round(1)

In [ ]:
# Experiment 6: TTFT at one user vs prompt length, per engine
pl = analyze.load(RESULTS_DIR, 'prompt_length')
one = pl[pl.concurrency == 1]
analyze.metric_vs(one, 'isl', 'ttft_p50_ms', 'engine', 'TTFT p50 at 1 user (ms)', 'Prompt tokens',
                  logx=True, save=f'{RESULTS_DIR}/prompt_length_ttft.png')
piv = one.pivot_table(index='isl', columns='engine', values='ttft_p50_ms')
piv['vllm_minus_hf_ms'] = piv['vllm'] - piv['hf']
piv.round(1)   # gap growing with isl -> attention kernel; flat gap -> fixed overhead

In [ ]:
# Experiment 7: model size. Expected decode floor = weight bytes / 320 GB/s (T4 peak bandwidth)
ms = analyze.load(RESULTS_DIR, 'model_size')
params_b = {'qwen2.5_0.5b': 0.494, 'qwen2.5_1.5b': 1.544, 'qwen2.5_3b': 3.086}
ms['weights_gb'] = ms.config.map(params_b) * 2           # fp16 = 2 bytes/param
ms['bandwidth_floor_ms'] = ms.weights_gb / 320 * 1000
ms['achieved_GBps'] = ms.weights_gb / (ms.itl_p50_ms / 1000)
ms[['config','concurrency','weights_gb','itl_p50_ms','bandwidth_floor_ms','achieved_GBps',
    'output_tok_per_s','kv_capacity_tokens']].round(1)

In [ ]:
# Experiment 8: admission control at C=100
ac = analyze.load(RESULTS_DIR, 'admission_control')
ac['cap'] = ac.config.str.replace('max_seqs_', '').astype(int)
ac.sort_values('cap')[['config','ttft_p50_ms','ttft_p95_ms','itl_p50_ms','per_user_tok_per_s',
                       'e2e_p95_ms','output_tok_per_s','waiting_max']].round(1)

In [ ]:
# Memory: where the T4's 15 GB went, and why paging matters
import pandas as pd
from pathlib import Path
from infbench.server import VLLMServer
mem = [{'config': d.name, **VLLMServer('x', [], d / 'server.log').kv_capacity()}
       for d in sorted(Path(f'{RESULTS_DIR}/concurrency_sweep').iterdir()) if (d / 'server.log').exists()]
display(pd.DataFrame(mem))     # parsed from each config's server.log

# Fragmentation: a server that reserves max_model_len of KV per request vs vLLM's 16-token pages
bytes_per_tok = 28 * 2 * 2 * 128 * 2       # layers x (K,V) x kv_heads x head_dim x fp16 = 28,672 B
cap = int(sweep[sweep.config == 'cb_fp16'].kv_capacity_tokens.iloc[0])
used, max_len, block = 512 + 128, 2048, 16
print(f"KV per token: {bytes_per_tok/1024:.0f} KiB | KV pool: {cap*bytes_per_tok/2**30:.2f} GiB = {cap:,} tokens")
print(f"Reserve-max-length server: {cap // max_len} requests fit, {100*(1-used/max_len):.0f}% of each reservation unused")
print(f"Paged (vLLM):              {cap // used} requests fit, waste < 1 block = {100*(block-1)/used:.1f}% worst case")

## 16 · Turning numbers into findings
Answer each with a number from the tables, then the *why*:

1. **Where is the knee?** At what concurrency does `cb_fp16` throughput stop growing while ITL keeps rising?
   (Decode is memory-bandwidth bound: each step reads all weights once, so more sequences per step are
   almost free — until the step becomes compute-bound or the KV cache fills.)
2. **Engine vs batching:** split the total gain into two steps — `baseline_hf_naive` → `vllm_no_batching`
   (engine: CUDA graphs, fused kernels) and `vllm_no_batching` → `cb_fp16` (batching). Where does the
   baselines' latency come from? (TTFT ≈ queueing time — check `waiting_max`.)
3. **KV-cache capacity:** `kv_capacity_tokens` for `cb_fp16` vs `cb_fp16_small_kv`. At C=100 × 640 tokens,
   does demand exceed capacity? What did that do to TTFT p95 and preemptions?
4. **Quantization:** AWQ frees ~2 GB of weights → more KV capacity. Is single-user ITL faster or slower
   than FP16 on the T4 (dequantisation cost vs fewer bytes read)? Does that change at C=100?
5. **Prefix caching:** TTFT reduction and hit rate; why does it help prefill but not ITL?
6. **Static vs continuous:** throughput gap and P99 latency with variable output lengths.
7. **Generation stalls:** what share of token gaps are ~400 ms stalls at C=25/50/100, and how does the
   chunked-prefill budget trade ITL p99 against TTFT?
8. **Prefill kernel:** does vLLM's TTFT gap vs HF grow with prompt length (attention kernel) or stay flat?
9. **Model size:** achieved GB/s per model — does the 0.5B model hit an overhead floor?
10. **Admission control:** which `max_num_seqs` gives the best per-user speed without exploding TTFT?
11. **Memory:** weights vs KV vs CUDA graphs, and how many requests fit with paging vs max-length reservations.
12. **Measurement caveats:** `gpu_util_pct` is ~100% even at 1 user, because NVML counts time *any* kernel
    runs, not how busy the GPU is. Use achieved GB/s and tok/s instead. CPU was ~30%, so not a bottleneck.